In [1]:
!pip install kagglehub

import kagglehub

# Correct dataset: grassknoted/asl-alphabet
path = kagglehub.dataset_download("grassknoted/asl-alphabet")

print("Dataset Downloaded to:", path)

Using Colab cache for faster access to the 'asl-alphabet' dataset.
Dataset Downloaded to: /kaggle/input/asl-alphabet


In [2]:
import os

DATASET_PATH = "/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train"
if os.path.exists(DATASET_PATH):
    classes = sorted(os.listdir(DATASET_PATH))
    print(f"Total Classes: {len(classes)}")
    print("Classes:", classes)

Total Classes: 29
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [4]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 9.7 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [5]:
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Download MediaPipe Hand Landmarker model bundle
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

# Setup Task Landmarker options
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)

DATASET_PATH = "/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train"
classes = sorted(os.listdir(DATASET_PATH))

X_data = []
y_data = []
IMAGES_PER_CLASS = 500  # Fast processing සඳහා class එකකට images 500ක්

print("Extracting Landmarks...")

for class_name in classes:
    class_dir = os.path.join(DATASET_PATH, class_name)
    image_files = os.listdir(class_dir)[:IMAGES_PER_CLASS]
    print(f"Processing Class: {class_name}")

    for img_name in image_files:
        img_path = os.path.join(class_dir, img_name)

        # Load image via MediaPipe Image object
        mp_image = mp.Image.create_from_file(img_path)
        detection_result = detector.detect(mp_image)

        if detection_result.hand_landmarks:
            hand_landmarks = detection_result.hand_landmarks[0]

            # Wrist Relative Normalization (Point 0 relative)
            wrist_x = hand_landmarks[0].x
            wrist_y = hand_landmarks[0].y
            wrist_z = hand_landmarks[0].z

            landmarks = []
            for lm in hand_landmarks:
                landmarks.extend([
                    lm.x - wrist_x,
                    lm.y - wrist_y,
                    lm.z - wrist_z
                ])

            X_data.append(landmarks)
            y_data.append(class_name)

X_data = np.array(X_data)
y_data = np.array(y_data)

print(f"\nExtraction Completed Successfully!")
print(f"Total Extracted Samples: {X_data.shape[0]}")

Extracting Landmarks...
Processing Class: A
Processing Class: B
Processing Class: C
Processing Class: D
Processing Class: E
Processing Class: F
Processing Class: G
Processing Class: H
Processing Class: I
Processing Class: J
Processing Class: K
Processing Class: L
Processing Class: M
Processing Class: N
Processing Class: O
Processing Class: P
Processing Class: Q
Processing Class: R
Processing Class: S
Processing Class: T
Processing Class: U
Processing Class: V
Processing Class: W
Processing Class: X
Processing Class: Y
Processing Class: Z
Processing Class: del
Processing Class: nothing
Processing Class: space

Extraction Completed Successfully!
Total Extracted Samples: 10615


In [7]:
import pandas as pd

# 1. made DataFrame
df = pd.DataFrame(X_data)
df['label'] = y_data


columns = []
for i in range(21):
    columns.extend([f'lm{i}_x', f'lm{i}_y', f'lm{i}_z'])
columns.append('label')

df.columns = columns

# 3. Updated CSV Save
df.to_csv("asl_hand_landmarks.csv", index=False)

print("Updated asl_hand_landmarks.csv file saved successfully with landmark column names!")

Updated asl_hand_landmarks.csv file saved successfully with landmark column names!


In [8]:
df.head()

,lm0_x,lm0_y,lm0_z,lm1_x,lm1_y,lm1_z,lm2_x,lm2_y,lm2_z,lm3_x,...,lm18_x,lm18_y,lm18_z,lm19_x,lm19_y,lm19_z,lm20_x,lm20_y,lm20_z,label
0,0.0,0.0,0.0,0.072259,-0.060622,-0.023509,0.116392,-0.159369,-0.031421,0.101213,...,-0.130518,-0.187882,-0.070202,-0.095215,-0.140317,-0.064767,-0.072038,-0.098081,-0.051569,A
1,0.0,0.0,0.0,0.092672,-0.022596,-0.013421,0.151948,-0.108716,-0.015186,0.161855,...,-0.045885,-0.189659,-0.076672,-0.031178,-0.137909,-0.074687,-0.024995,-0.095191,-0.065043,A
2,0.0,0.0,0.0,0.113663,-0.062687,-0.020164,0.180032,-0.175017,-0.028697,0.196123,...,-0.058516,-0.239883,-0.095678,-0.045303,-0.178426,-0.087779,-0.042958,-0.127539,-0.072697,A
3,0.0,0.0,0.0,0.085950,-0.032255,-0.017578,0.157735,-0.107470,-0.025790,0.169868,...,-0.027946,-0.216185,-0.072926,-0.022874,-0.164330,-0.063680,-0.029088,-0.122640,-0.050525,A
4,0.0,0.0,0.0,0.105170,-0.033261,-0.014146,0.171582,-0.118460,-0.017291,0.185673,...,-0.027456,-0.209453,-0.070097,-0.017025,-0.152620,-0.066813,-0.014212,-0.108602,-0.056030,A
